In [2]:
import os
print(os.getcwd())

c:\Users\Shravni Joshi\Documents\Diabetic Retinopathy\diabetic-retinopathy-xai\notebooks


In [3]:
!pip install captum


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Debug cell — let's actually SEE what's going on with paths and the xai folder
import sys
from pathlib import Path

print("Current working directory:", Path.cwd())
print("\nsys.path entries:")
for p in sys.path:
    print(f"  {p}")

repo_root_guess = Path("..").resolve()
print(f"\nLooking inside: {repo_root_guess}")
print("Contents of repo root:")
for item in repo_root_guess.iterdir():
    print(f"  {item.name}{'/' if item.is_dir() else ''}")

xai_path = repo_root_guess / "xai"
print(f"\nDoes {xai_path} exist? {xai_path.exists()}")
if xai_path.exists():
    print("Contents of xai/ at repo root:")
    for item in xai_path.iterdir():
        print(f"  {item.name}")

Current working directory: c:\Users\Shravni Joshi\Documents\Diabetic Retinopathy\diabetic-retinopathy-xai\notebooks

sys.path entries:
  c:\Users\Shravni Joshi\AppData\Local\Programs\Python\Python313\python313.zip
  c:\Users\Shravni Joshi\AppData\Local\Programs\Python\Python313\DLLs
  c:\Users\Shravni Joshi\AppData\Local\Programs\Python\Python313\Lib
  c:\Users\Shravni Joshi\AppData\Local\Programs\Python\Python313
  
  C:\Users\Shravni Joshi\AppData\Roaming\Python\Python313\site-packages
  c:\Users\Shravni Joshi\AppData\Local\Programs\Python\Python313\Lib\site-packages
  c:\Users\Shravni Joshi\AppData\Local\Programs\Python\Python313\Lib\site-packages\win32
  c:\Users\Shravni Joshi\AppData\Local\Programs\Python\Python313\Lib\site-packages\win32\lib
  c:\Users\Shravni Joshi\AppData\Local\Programs\Python\Python313\Lib\site-packages\Pythonwin
  ..
  ..
  ..
  ..
  ..
  ..
  ..
  ..
  ..
  ..
  ..

Looking inside: C:\Users\Shravni Joshi\Documents\Diabetic Retinopathy\diabetic-retinopathy-xa

In [11]:
# Cell 1 — Imports + Paths
# Just setup, nothing executes any real logic yet. If this cell fails,
# it's almost certainly a missing package or wrong path — fix here first.

import sys
import json
from pathlib import Path

import torch
import pandas as pd

# Add repo root to path so we can import from xai/
repo_root_str = str(Path("..").resolve())
if repo_root_str not in sys.path:
    sys.path.append(repo_root_str)

from xai.shap_wrapper import (
    SHAPWrapper,
    build_background_tensor,
    load_image_tensor,
    normalize_and_resize,
    save_heatmap,
    set_seed,
)

# ---------------------------------------------------------------------------
# Paths — LOCAL VS Code setup.
# CODE lives in the git-cloned repo. DATA + CHECKPOINTS live on the shared
# Google Drive (gitignored, too large for git), synced locally via
# Google Drive for Desktop in stream mode.
# ---------------------------------------------------------------------------
REPO_ROOT = Path("..").resolve()  # the git-cloned repo (code only)

DRIVE_ROOT = Path(
    r"G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai"
)
DATA_ROOT = DRIVE_ROOT / "data" / "IDRiD"
CKPT_DIR = DRIVE_ROOT / "results" / "checkpoints"

GRADING_TEST_DIR = DATA_ROOT / "grading" / "images" / "test"
GRADING_TRAIN_DIR = DATA_ROOT / "grading" / "images" / "train"
TRAIN_CSV = DATA_ROOT / "grading" / "labels" / "train.csv"
TEST_CSV = DATA_ROOT / "grading" / "labels" / "test.csv"

# OUTPUT_ROOT: where YOUR generated heatmaps get saved. Since this is new
# output you're creating (not data you're reading), it's fine -- arguably
# better -- to save it into the LOCAL repo clone, then commit/push or share
# separately with the team. Keeping it local avoids slow writes over the
# Drive stream connection.
OUTPUT_ROOT = REPO_ROOT / "results" / "heatmaps"
TEST_IDS_PATH = REPO_ROOT / "test_image_ids.json"

print(f"REPO_ROOT resolved to: {REPO_ROOT}")
print(f"DRIVE_ROOT resolved to: {DRIVE_ROOT}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

SEED = 42
set_seed(SEED)

# ---------------------------------------------------------------------------
# Sanity print — confirm paths actually exist BEFORE we try to use them
# ---------------------------------------------------------------------------
paths_to_check = {
    "GRADING_TEST_DIR": GRADING_TEST_DIR,
    "GRADING_TRAIN_DIR": GRADING_TRAIN_DIR,
    "TRAIN_CSV": TRAIN_CSV,
    "TEST_CSV": TEST_CSV,
    "CKPT_DIR": CKPT_DIR,
    "TEST_IDS_PATH": TEST_IDS_PATH,
}

print("\nPath check:")
for name, p in paths_to_check.items():
    exists = Path(p).exists()
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {name}: {p}")

REPO_ROOT resolved to: C:\Users\Shravni Joshi\Documents\Diabetic Retinopathy\diabetic-retinopathy-xai
DRIVE_ROOT resolved to: G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai
Using device: cpu

Path check:
  [OK] GRADING_TEST_DIR: G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai\data\IDRiD\grading\images\test
  [OK] GRADING_TRAIN_DIR: G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai\data\IDRiD\grading\images\train
  [OK] TRAIN_CSV: G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai\data\IDRiD\grading\labels\train.csv
  [OK] TEST_CSV: G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai\data\IDRiD\grading\labels\test.csv
  [OK] CKPT_DIR: G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai\results\checkpoints
  [OK] TEST_IDS_PATH: C:\Users\Shravni Joshi\Documents\Diabetic Retino

In [ ]:
# Debug cell — what's ACTUALLY inside GRADING_TEST_DIR?
# Run this instead of guessing the filename pattern.

contents = list(GRADING_TEST_DIR.iterdir())
print(f"Total items in folder: {len(contents)}")
print("\nFirst 10 items:")
for item in contents[:10]:
    print(f"  {item.name}")

Total items in folder: 103

First 10 items:
  IDRiD_087.jpg
  IDRiD_090.jpg
  IDRiD_088.jpg
  IDRiD_091.jpg
  IDRiD_089.jpg
  IDRiD_001.jpg
  IDRiD_002.jpg
  IDRiD_003.jpg
  IDRiD_004.jpg
  IDRiD_005.jpg


In [ ]:
# Debug cell — what's the FULL range of IDs in this folder?
# We need to know if this is really the 27-image official test split,
# or if it's actually a bigger folder (all images, or train+test mixed).

import re

contents = sorted(GRADING_TEST_DIR.iterdir())
ids_found = []
for item in contents:
    match = re.search(r"IDRiD_(\d+)", item.stem)
    if match:
        ids_found.append(int(match.group(1)))

print(f"Total files: {len(contents)}")
print(f"ID range: {min(ids_found)} to {max(ids_found)}")
print(f"All IDs: {sorted(ids_found)}")

# Cross-check: does this folder actually contain IDRiD_55 through IDRiD_81
# (the frozen test set from test_image_ids.json), just zero-padded?
expected_numbers = [int(tid.split("_")[1]) for tid in test_image_ids]
print(f"\nExpected (from test_image_ids.json): {sorted(expected_numbers)}")

missing = [n for n in expected_numbers if n not in ids_found]
print(f"\nMissing from folder: {missing}")

Total files: 103
ID range: 1 to 103
All IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103]

Expected (from test_image_ids.json): [55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81]

Missing from folder: []


In [12]:
# Cell 2 — Sanity check: load frozen test IDs + load ONE image
# Just confirms file paths resolve to real, readable images before we
# touch any model or SHAP logic.

with open(TEST_IDS_PATH) as f:
    test_image_ids = json.load(f)

print(f"Loaded {len(test_image_ids)} frozen test image IDs.")
print(f"First few: {test_image_ids[:5]}")
print(f"Last few: {test_image_ids[-5:]}")

# Try loading just the FIRST test image to confirm load_image_tensor() works
#
# NOTE: test_image_ids.json stores IDs as "IDRiD_55" (no padding), but the
# actual files on disk are zero-padded to 3 digits: "IDRiD_055.jpg".
# We keep test_image_ids.json untouched (it's the locked, frozen contract)
# and pad ONLY when building the actual file path.

def id_to_filename(image_id: str) -> str:
    """'IDRiD_55' -> 'IDRiD_055.jpg' (3-digit zero-padded, matches folder)."""
    prefix, num = image_id.split("_")
    return f"{prefix}_{int(num):03d}.jpg"

first_id = test_image_ids[0]
first_image_path = GRADING_TEST_DIR / id_to_filename(first_id)

print(f"\nTrying to load: {first_image_path}")
print(f"File exists: {first_image_path.exists()}")

input_tensor, original_size = load_image_tensor(str(first_image_path))

print(f"\nLoaded tensor shape: {input_tensor.shape}")  # expect (1, 3, 512, 512)
print(f"Original image size (H, W): {original_size}")
print(f"Tensor dtype: {input_tensor.dtype}")
print(f"Tensor value range: [{input_tensor.min().item():.3f}, {input_tensor.max().item():.3f}]")

Loaded 27 frozen test image IDs.
First few: ['IDRiD_55', 'IDRiD_56', 'IDRiD_57', 'IDRiD_58', 'IDRiD_59']
Last few: ['IDRiD_77', 'IDRiD_78', 'IDRiD_79', 'IDRiD_80', 'IDRiD_81']

Trying to load: G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai\data\IDRiD\grading\images\test\IDRiD_055.jpg
File exists: True

Loaded tensor shape: torch.Size([1, 3, 512, 512])
Original image size (H, W): (2848, 4288)
Tensor dtype: torch.float32
Tensor value range: [-2.118, 2.249]


In [ ]:
# Debug cell — what checkpoint files actually exist?
ckpt_contents = list(CKPT_DIR.iterdir())
print(f"Found {len(ckpt_contents)} items in checkpoints folder:")
for item in ckpt_contents:
    size_mb = item.stat().st_size / (1024 * 1024)
    print(f"  {item.name}  ({size_mb:.1f} MB)")

Found 4 items in checkpoints folder:
  efficientnet_b4_best.pth  (67.7 MB)
  resnet50_best.pth  (90.0 MB)
  training_history.json  (0.0 MB)
  desktop.ini  (0.0 MB)


In [ ]:
pip install timm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Cell 3 — Sanity check: load ONE checkpoint (EfficientNet-B4 first)
# This is the step most likely to surface a real mismatch. If it fails,
# the error message will usually tell us exactly what doesn't match
# (missing keys, unexpected keys, shape mismatch) — read it carefully,
# don't just retry blindly.

MODEL_CHECKPOINTS = {
    "efficientnetb4": CKPT_DIR / "efficientnet_b4_best.pth",
    "resnet50": CKPT_DIR / "resnet50_best.pth",
}

def load_model(model_name: str, checkpoint_path) -> torch.nn.Module:
    """
    Builds the architecture, then loads trained weights on top.

    IMPORTANT: EfficientNet-B4 checkpoint keys match `timm`'s naming
    convention (conv_stem, blocks.N.M, se.conv_reduce/expand, flat
    classifier.weight) -- NOT torchvision's (features.N, classifier.1).
    Confirmed by inspecting the actual state_dict keys in the error
    message from the failed torchvision load attempt. ResNet-50 key
    names are usually identical between timm and torchvision, but we
    use timm for both here for consistency -- if ResNet-50 loading
    fails too, we'll adjust.
    """
    import timm

    if model_name == "efficientnetb4":
        model = timm.create_model("efficientnet_b4", pretrained=False, num_classes=5)
    elif model_name == "resnet50":
        model = timm.create_model("resnet50", pretrained=False, num_classes=5)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    checkpoint = torch.load(str(checkpoint_path), map_location=DEVICE)

    # Checkpoints can be saved a few different ways -- handle the common cases
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint  # assume it's a raw state_dict

    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()
    return model


print("Attempting to load EfficientNet-B4 checkpoint...")
print(f"Path: {MODEL_CHECKPOINTS['efficientnetb4']}")
print(f"File exists: {MODEL_CHECKPOINTS['efficientnetb4'].exists()}")

effnet_model = load_model("efficientnetb4", MODEL_CHECKPOINTS["efficientnetb4"])
print("\nSUCCESS — EfficientNet-B4 loaded without errors.")
print(f"Model is in eval mode: {not effnet_model.training}")

Attempting to load EfficientNet-B4 checkpoint...
Path: G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai\results\checkpoints\efficientnet_b4_best.pth
File exists: True

SUCCESS — EfficientNet-B4 loaded without errors.
Model is in eval mode: True


In [14]:
# Cell 4 — Sanity check: load ResNet-50 checkpoint too
# Same approach as cell 3, just the second model. If EfficientNet-B4
# needed timm, ResNet-50 probably does too (for consistency with how
# your friend likely trained both), but let's confirm rather than assume.

print("Attempting to load ResNet-50 checkpoint...")
print(f"Path: {MODEL_CHECKPOINTS['resnet50']}")
print(f"File exists: {MODEL_CHECKPOINTS['resnet50'].exists()}")

resnet_model = load_model("resnet50", MODEL_CHECKPOINTS["resnet50"])
print("\nSUCCESS — ResNet-50 loaded without errors.")
print(f"Model is in eval mode: {not resnet_model.training}")

Attempting to load ResNet-50 checkpoint...
Path: G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai\results\checkpoints\resnet50_best.pth
File exists: True

SUCCESS — ResNet-50 loaded without errors.
Model is in eval mode: True


In [15]:
# Cell 5 — Sanity check: does the model actually predict something sensible?
# Uses the SHAPWrapper.predict() method (no SHAP yet, just a forward pass +
# softmax). Confirms model + image + device all talk to each other correctly.

# We need a background tensor to even construct SHAPWrapper, but predict()
# doesn't actually use it -- so a tiny dummy placeholder is fine here just
# to test prediction. Real background comes in cell 6.
dummy_background = input_tensor.clone()  # shape (1, 3, 512, 512), just a placeholder

wrapper = SHAPWrapper(
    model=effnet_model,
    background=dummy_background,
    device=DEVICE,
)

predicted_class, confidence = wrapper.predict(input_tensor)

print(f"Image: {first_id}")
print(f"Predicted DR grade: {predicted_class}")
print(f"Confidence: {confidence:.4f}")

# Cross-check against ground truth, just out of curiosity (not required,
# just a sanity signal -- model doesn't need to be RIGHT, just needs to
# produce a believable class 0-4 with a believable confidence, not NaN
# or garbage)
test_df = pd.read_csv(TEST_CSV)
print(f"\nTEST_CSV columns: {list(test_df.columns)}")

Image: IDRiD_55
Predicted DR grade: 0
Confidence: 0.4475

TEST_CSV columns: ['Image name', 'Retinopathy grade', 'Risk of macular edema ']


In [16]:
# Cell 6 — Build the FIXED Grade-0 background sample
# This background is used for EVERY explain() call, for BOTH models.
# Built once here, reused throughout — not resampled per image.

train_df = pd.read_csv(TRAIN_CSV)
print(f"TRAIN_CSV columns: {list(train_df.columns)}")
print(f"Total training images: {len(train_df)}")

grade0_df = train_df[train_df["Retinopathy grade"] == 0]
print(f"\nGrade-0 (no DR) training images available: {len(grade0_df)}")

# Same zero-padding fix as test images -- apply id_to_filename() when
# building actual file paths, but keep the CSV's "Image name" values
# (e.g. "IDRiD_001" or "IDRiD_1") untouched.
#
# NOTE: we don't actually know yet whether train.csv's "Image name" column
# already has the zero-padded form or the short form -- print a few to check
# before assuming id_to_filename() is even needed here.
print(f"\nSample 'Image name' values from train.csv:")
print(grade0_df["Image name"].head(5).tolist())

TRAIN_CSV columns: ['Image name', 'Retinopathy grade', 'Risk of macular edema ', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']
Total training images: 413

Grade-0 (no DR) training images available: 134

Sample 'Image name' values from train.csv:
['IDRiD_118', 'IDRiD_138', 'IDRiD_139', 'IDRiD_140', 'IDRiD_141']


In [17]:
# Cell 7 — Build the actual fixed background tensor
# train.csv "Image name" values are ALREADY zero-padded (e.g. "IDRiD_118"),
# matching the file naming convention directly. No padding fix needed here
# (unlike the test set, where test_image_ids.json had unpadded IDs).

grade0_image_names = grade0_df["Image name"].tolist()
grade0_paths = [str(GRADING_TRAIN_DIR / f"{name}.jpg") for name in grade0_image_names]

# Quick check before committing — does the first path actually exist?
print(f"First Grade-0 path: {grade0_paths[0]}")
print(f"Exists: {Path(grade0_paths[0]).exists()}")

N_BACKGROUND = 20

background_tensor = build_background_tensor(
    grade0_paths, n_background=N_BACKGROUND, seed=SEED, device=DEVICE
)

print(f"\nBackground tensor shape: {background_tensor.shape}")  # expect (20, 3, 512, 512)
print(f"Background tensor dtype: {background_tensor.dtype}")

First Grade-0 path: G:\.shortcut-targets-by-id\1pDMOv8oZXnC8AiUIUxcu0OaihyxWM146\diabetic-retinopathy-xai\data\IDRiD\grading\images\train\IDRiD_118.jpg
Exists: True

Background tensor shape: torch.Size([20, 3, 512, 512])
Background tensor dtype: torch.float32


In [18]:
# Cell 8 (RETRY #2) — even more conservative for 8GB RAM.
#
# Changes from the previous crash:
# - n_samples: 50 -> 3 (even fewer)
# - background size: 20 -> 3 (even smaller)
# - explicit garbage collection before running, to free up whatever RAM
#   we can before this memory-heavy operation
#
# Goal here is just: does ANYTHING complete without crashing? Once we
# know that, we step up gradually from a known-working floor.

import gc
import time
import numpy as np

gc.collect()  # free up memory from anything lingering

tiny_background = background_tensor[:3]
print(f"Diagnostic background shape: {tiny_background.shape}")

diagnostic_wrapper = SHAPWrapper(
    model=effnet_model,
    background=tiny_background,
    device=DEVICE,
    n_samples=3,
)

print("Running GradientSHAP with MINIMAL settings (n_samples=3, background=3)...")
print("Watch Task Manager now if you can.")
start_time = time.time()

raw_heatmap = diagnostic_wrapper.explain(input_tensor, target_class=predicted_class)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f} seconds.")
print(f"Raw heatmap shape: {raw_heatmap.shape}")
print(f"Value range: [{raw_heatmap.min():.6f}, {raw_heatmap.max():.6f}]")
print(f"Any NaN: {np.isnan(raw_heatmap).any()}")

Diagnostic background shape: torch.Size([3, 3, 512, 512])
Running GradientSHAP with MINIMAL settings (n_samples=3, background=3)...
Watch Task Manager now if you can.

Done in 79.6 seconds.
Raw heatmap shape: (512, 512)
Value range: [-0.802576, 0.781694]
Any NaN: False


In [19]:
# Diagnostic — isolate where time is ACTUALLY going.
# Before trying another fix blindly, let's measure:
# 1. How long does ONE plain forward pass through the model take?
# 2. How long does ONE forward+backward (gradient) pass take?
# This tells us whether the model itself is the bottleneck, or whether
# it's specifically captum's sampling/perturbation overhead.

import time
import torch

# 1. Plain forward pass (no gradients), full 512x512, real model directly
with torch.no_grad():
    start = time.time()
    _ = effnet_model(input_tensor.to(DEVICE))
    elapsed_forward = time.time() - start
print(f"Single forward pass (512x512, no grad): {elapsed_forward:.3f} sec")

# 2. Forward + backward pass (gradient computation), full 512x512
input_grad_test = input_tensor.clone().to(DEVICE)
input_grad_test.requires_grad_()

start = time.time()
output = effnet_model(input_grad_test)
loss = output[0, predicted_class]
loss.backward()
elapsed_backward = time.time() - start
print(f"Single forward+backward pass (512x512): {elapsed_backward:.3f} sec")

# Rough extrapolation: GradientShap with n_samples=N internally does
# roughly N forward+backward passes per call (plus baseline expansion
# overhead). Let's see if that math roughly matches what we observed.
print(f"\nRough estimate for n_samples=3: ~{elapsed_backward * 3:.1f} sec")
print(f"Rough estimate for n_samples=5: ~{elapsed_backward * 5:.1f} sec")
print("(Actual observed: n_samples=3 -> 51.4s, n_samples=5 -> 166.0s)")
print("If actual >> estimate, overhead is in captum's sampling/memory, not raw compute.")

Single forward pass (512x512, no grad): 0.932 sec
Single forward+backward pass (512x512): 3.978 sec

Rough estimate for n_samples=3: ~11.9 sec
Rough estimate for n_samples=5: ~19.9 sec
(Actual observed: n_samples=3 -> 51.4s, n_samples=5 -> 166.0s)
If actual >> estimate, overhead is in captum's sampling/memory, not raw compute.


In [9]:
# Cell 9 — Test how n_samples affects speed (background fixed at 3 for now)
# Running n_samples = 5, then 10, then 20, one at a time, watching elapsed
# time after each. Run this cell, note the time, THEN decide whether to
# continue to the next value or stop here -- don't blindly run all three
# back to back if memory feels tight.

import gc
import time
import numpy as np

gc.collect()

N_SAMPLES_TEST = 5  # change this value and re-run to test 5, then 10, then 20

test_wrapper = SHAPWrapper(
    model=effnet_model,
    background=background_tensor[:3],  # keep background fixed at 3 for this test
    device=DEVICE,
    n_samples=N_SAMPLES_TEST,
)

print(f"Testing n_samples={N_SAMPLES_TEST}, background=3...")
start_time = time.time()

raw_heatmap = test_wrapper.explain(input_tensor, target_class=predicted_class)

elapsed = time.time() - start_time
print(f"Done in {elapsed:.1f} seconds.")
print(f"Value range: [{raw_heatmap.min():.6f}, {raw_heatmap.max():.6f}]")
print(f"Any NaN: {np.isnan(raw_heatmap).any()}")

Testing n_samples=5, background=3...
Done in 166.0 seconds.
Value range: [-0.339620, 0.324901]
Any NaN: False
